In [1]:
from transformers import AutoTokenizer
import os
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from collections import defaultdict
import pandas as pd
import json
from collections import defaultdict


/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

## WOS

In [107]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
source = []
labels = []
label_ids = []
label_dict = {}
hiera = defaultdict(set)
with open('/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/WebOfScience/wos_total.json', 'r') as f:
    for line in f.readlines():
        line = json.loads(line)
        source.append(tokenizer.encode(line['doc_token'].strip().lower(), truncation=True))
        labels.append(line['doc_label'])
for l in labels:
    if l[0] not in label_dict:
        label_dict[l[0]] = len(label_dict)
for l in labels:
    assert len(l) == 2
    if l[1] not in label_dict:
        label_dict[l[1]] = len(label_dict)
    label_ids.append([label_dict[l[0]], label_dict[l[1]]])
    hiera[label_ids[-1][0]].add(label_ids[-1][1])
value_dict = {i: tokenizer.encode(v.lower(), add_special_tokens=False) for v, i in label_dict.items()}

In [110]:
label_ids[-1]

[4, 13]

In [115]:
hiera[0]

{7, 12, 23, 28, 37, 41, 46, 49, 50, 57, 91, 99, 101, 110, 111, 125, 140}

## PaNET

In [71]:
import joblib
from owlready2 import *


# Load the ontology
ontology_path='/Users/fdp54928/Documents/PaNET-classifier/Data/owlapi.xrdf'
onto=get_ontology(ontology_path).load()

# Run reasoner
with onto:
    sync_reasoner()  # Runs reasoning and updates inferred relationships

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/venv/lib/python3.11/site-packages/owlready2/hermit:/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/venv/lib/python3.11/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/jt/qsn_d43943dbh0h9ngcrb3n80000gq/T/tmp9p0zi787
* Owlready2 * HermiT took 0.44919610023498535 seconds
* Owlready * Equivalenting: PaNET.PaNET01022 wiki.Q133900
* Owlready * Equivalenting: wiki.Q133900 PaNET.PaNET01022
* Owlready * Equivalenting: PaNET.PaNET01107 PaNET.PaNET01273
* Owlready * Equivalenting: PaNET.PaNET01273 PaNET.PaNET01107
* Owlready * Equivalenting: PaNET.PaNET2014001 PaNET.PaNET2020070
* Owlready * Equivalenting: PaNET.PaNET2020070 PaNET.PaNET2014001
* Owlready * Equivalenting: PaNET.PaNET00401 PaNET.PaNET01106
* Owlready * Equivalenting: PaNET.PaNET011

In [ ]:
root = onto.search_one(iri='http://purl.org/pan-science/PaNET/PaNET00001')
panet_dict = {}
for cls in root.descendants():
    if cls.iri is not None and len(cls.label) > 0:
        panet_dict[cls.iri] = cls.label[0]
    else:
        print(f"Warning: Class {cls} has no label or IRI.")

In [ ]:
import json

with open('/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/iri_to_label_dict.json','w') as f:
    json.dump(panet_dict, f, indent=4)

In [102]:
with open('/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/iri_to_label_dict.json') as f:
    panet_dict = json.load(f)

In [85]:
def save_processed_file(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')

import json

def load_processed_file(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # Strip whitespace and ignore empty lines
            if line.strip():
                data.append(json.loads(line))
    return data


def convert_to_label(ls_of_dict, panet_dict):
    full_ls = []
    for d in ls_of_dict:
        ls = []
        for iri in d['label']:
            ls.append(panet_dict[iri])
        d['label'] = ls
        full_ls.append(d)
    return full_ls

In [86]:
datapath = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/'
test = load_processed_file(datapath + 'test_raw.json')
test = convert_to_label(test, panet_dict)
save_processed_file(test, datapath + 'test.json')

datapath = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/'
train = load_processed_file(datapath + 'train_raw.json')
train = convert_to_label(train, panet_dict)
save_processed_file(train, datapath + 'train.json')

datapath = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/'
eval = load_processed_file(datapath + 'eval_raw.json')
eval = convert_to_label(eval, panet_dict)
save_processed_file(eval, datapath + 'eval.json')

In [164]:
from datasets import Dataset, DatasetDict


In [ ]:
datapath = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/test_raw.json'
binarizer = joblib.load('/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/binarizer.pkl')

source = []
labels = []

with open(datapath, 'r') as f:
    for line in f.readlines():
        line = json.loads(line)
        source.append(tokenizer.encode(line['token'].strip().lower(), truncation=True))
        labels.append(line['label'])

# Use the binarizer directly — consistent with how train/test split was originally encoded
one_hot_labels = binarizer.transform(labels).tolist()


# --- Create HuggingFace Dataset ---
test_dataset = Dataset.from_dict({
    "input_ids": source,          # list of token id lists (variable length)
    "labels": one_hot_labels,     #àlist of one-hot int lists
})


# dataset = DatasetDict({
#     "test": test_dataset
# })

In [176]:
datapath = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/train_raw.json'
binarizer = joblib.load('/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/binarizer.pkl')

source = []
labels = []

with open(datapath, 'r') as f:
    for line in f.readlines():
        line = json.loads(line)
        source.append(tokenizer.encode(line['token'].strip().lower(), truncation=True))
        labels.append(line['label'])

# Use the binarizer directly — consistent with how train/test split was originally encoded
one_hot_labels = binarizer.transform(labels).tolist()


# --- Create HuggingFace Dataset ---
train_dataset = Dataset.from_dict({
    "input_ids": source,          # list of token id lists (variable length)
    "labels": one_hot_labels,     #àlist of one-hot int lists
})

In [177]:
datapath = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/eval_raw.json'
binarizer = joblib.load('/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/binarizer.pkl')

source = []
labels = []

with open(datapath, 'r') as f:
    for line in f.readlines():
        line = json.loads(line)
        source.append(tokenizer.encode(line['token'].strip().lower(), truncation=True))
        labels.append(line['label'])

# Use the binarizer directly — consistent with how train/test split was originally encoded
one_hot_labels = binarizer.transform(labels).tolist()


# --- Create HuggingFace Dataset ---
eval_dataset = Dataset.from_dict({
    "input_ids": source,          # list of token id lists (variable length)
    "labels": one_hot_labels,     #àlist of one-hot int lists
})

In [178]:
train = list(range(len(train_dataset)))
val = list(range(len(train), len(eval_dataset)+len(train)))
test = list(range(len(train)+len(val), len(test)+len(train)+len(val)))

In [184]:
test_dataset

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 1187
})

In [131]:
value_dict2 = {i: tokenizer.encode(panet_dict[v].lower(), add_special_tokens=False)
        for i, v in enumerate(binarizer.classes_)}

In [116]:
path = '/Users/fdp54928/Library/CloudStorage/OneDrive-Nexus365/GitHub Repositories/contrastive-htc/data/panet/'
subclass_map = {}

with open(f"{path}panet.taxonomy", 'r', encoding='utf-8') as f:
    for line in f:
        # Strip the newline and split by tabs
        parts = line.strip().split('\t')
        
        if parts:
            parent = parts[0]
            # The rest of the elements (if any) are the children
            children = parts[1:]
            subclass_map[parent] = children

In [144]:
panet_idx_dict = {v: i for i, v in enumerate(binarizer.classes_)}

In [160]:
hiera2 = defaultdict(set)


In [161]:
for i, v in enumerate(binarizer.classes_):
    for child in subclass_map[v]:
        if child in panet_idx_dict:
            hiera2[i].add(panet_idx_dict[child])

In [185]:
from datasets import Dataset, load_from_disk


## Test

In [ ]:
# from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
import datasets
# import argparse
# import os
# from train import BertDataset
# from eval import evaluate
# from model.contrast import ContrastModel

/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/model/optim.py:62: SyntaxWarning: invalid escape sequence '\:'
  """Implements Adam algorithm.


ModuleNotFoundError: No module named 'torch_geometric'

In [2]:
split = torch.load('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/processed_data/split.pt')


In [7]:
test = Subset(dataset, split['test'])
test = DataLoader(test, batch_size=batch_size, shuffle=False, collate_fn=dataset.collate_fn)

NameError: name 'dataset' is not defined

In [6]:
split['test'][0]

10315

In [15]:
label_dict = torch.load('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/processed_data/bert_value_dict.pt')

In [16]:
len(label_dict)

140

In [18]:
from datasets import load_from_disk


In [20]:
dataset = load_from_disk('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/processed_data/')
# sdata = dataset['input_ids']
# labels = dataset['labels']


# self.data = data_utils.load_indexed_dataset(
#     data_path + '/tok', None, 'mmap'
# )
# self.labels = data_utils.load_indexed_dataset(
#     data_path + '/Y', None, 'mmap'
# )
# self.max_token = max_token
# self.pad_idx = pad_idx

        
# # Ensure data is a 1D tensor of Longs
# data = torch.tensor(self.data[item], dtype=torch.long)[:self.max_token - 2]

# # CRITICAL CHANGE: 
# # 1. Convert to Long (to match == 1)
# # 2. .view(-1) forces it to be a flat 1D vector of class flags [0, 1, 0...]
# labels = torch.tensor(self.labels[item], dtype=torch.long).view(-1).to(self.device)

In [27]:
dataset['labels'][-1]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

11501

In [23]:
test = Subset(dataset, split['test'])


In [24]:
test

In [28]:
from transformers import AutoTokenizer
# import os
import torch
import numpy as np
# from sklearn.model_selection import train_test_split
from collections import defaultdict
import pandas as pd
import json
from collections import defaultdict
from datasets import Dataset, DatasetDict
import joblib


In [29]:
def preprocess_data(datapath, tokenizer, binarizer):
    source = []
    labels = []

    with open(datapath, 'r') as f:
        for line in f.readlines():
            line = json.loads(line)
            source.append(tokenizer.encode(line['token'].strip().lower(), truncation=True))
            labels.append(line['label'])
    
    # Use the binarizer directly — consistent with how train/test split was originally encoded
    one_hot_labels = binarizer.transform(labels).tolist()

    
    # --- Create HuggingFace Dataset ---
    full_dataset = Dataset.from_dict({
        "input_ids": source,          # list of token id lists (variable length)
        "labels": one_hot_labels,     # list of one-hot int lists
    })

    return full_dataset


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
binarizer = joblib.load('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/binarizer.pkl')

# panet_dict => mapping from label id to label name
with open('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/iri_to_label_dict.json') as f:
    panet_dict = json.load(f)

# subclass_map => parent to children mapping
path = '/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/'
subclass_map = {}
with open(f"{path}panet.taxonomy", 'r', encoding='utf-8') as f:
    for line in f:
        # Strip the newline and split by tabs
        parts = line.strip().split('\t')
        if parts:
            parent = parts[0]
            # The rest of the elements (if any) are the children
            children = parts[1:]
            subclass_map[parent] = children

train_dataset = preprocess_data('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/train_raw.json', tokenizer, binarizer)
test_dataset = preprocess_data('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/test_raw.json', tokenizer, binarizer)
eval_dataset = preprocess_data('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/eval_raw.json', tokenizer, binarizer)
full_dataset = preprocess_data('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/full_data_raw.json', tokenizer, binarizer)


value_dict = {i: tokenizer.encode(panet_dict[v].lower(), add_special_tokens=False)
        for i, v in enumerate(binarizer.classes_)}

panet_idx_dict = {v: i for i, v in enumerate(binarizer.classes_)}
hiera = defaultdict(set)

for i, v in enumerate(binarizer.classes_):
    for child in subclass_map[v]:
        if child in panet_idx_dict:
            hiera[i].add(panet_idx_dict[child])


# torch.save(value_dict, 'processed_data/bert_value_dict.pt')
# torch.save(hiera, 'processed_data/slot.pt')


train, test, val = [], [], []

train = list(range(len(train_dataset)))
val = list(range(len(train), len(eval_dataset)+len(train)))
test = list(range(len(train)+len(val), len(test_dataset)+len(train)+len(val)))

# torch.save({'train': train, 'val': val, 'test': test}, 'processed_data/split.pt')


# full_dataset.save_to_disk('/Users/fdp54928/Documents/GitHub Repositories/contrastive-htc/data/panet/processed_data')

Saving the dataset (1/1 shards): 100%|██████████| 11502/11502 [00:00<00:00, 1206283.21 examples/s]
